# Discrete Mathematics — Chapter 4
## Elementary Number Theory and Methods of Proof
### Interactive companion notebook

**Course:** COMP 233 — Discrete Mathematics
**Textbook:** Susanna S. Epp, *Discrete Mathematics with Applications*, 4th Edition
**Sections covered:** 4.1, 4.2, 4.3, 4.4

---

### ⚠️ Important note about computational verification

Chapter 4 is about **proofs**, not computations. A computer can *test* a universal statement on many examples, but **finding no counterexample is not a proof** — there could always be one beyond the range you checked.

What computation *can* do well:
- Disprove a universal statement by **finding a counterexample**.
- Confirm an existential statement by **finding a witness**.
- Build intuition before writing a proof.
- Catch errors by sampling.

What computation *cannot* do:
- Prove a universal statement over an infinite domain.
- Replace the generic-particular method.

This notebook keeps this distinction visible at every step.

---

### How to use this notebook

1. Run cells top to bottom; widgets activate after their setup cell runs.
2. Each section ends with **3 practice exercises** with reveal-on-click solutions.

**Key Arabic terms** used in this notebook:
- البرهان (proof)
- المثال المضاد (counterexample)
- العدد الزوجي / العدد الفردي (even integer / odd integer)
- العدد النسبي / العدد غير النسبي (rational / irrational)
- القابلية للقسمة (divisibility)
- نظرية ال‍قسمة (quotient-remainder theorem)
- البرهان بالحالات (proof by cases)

---

In [ ]:
# Run this cell first.
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

import ipywidgets as widgets
from IPython.display import display
from math import gcd, isqrt
from fractions import Fraction

print("Setup complete.")

---
# Section 4.1 — Direct proof and counterexample

## 4.1.1 — Definitions (memorise word for word)

**Even** (*زوجي*).  An integer `n` is **even** if and only if `n = 2k` for some integer `k`.

**Odd** (*فردي*).  An integer `n` is **odd** if and only if `n = 2k + 1` for some integer `k`.

**Prime** (*أوّلي*).  An integer `n > 1` is **prime** if and only if its only positive divisors are 1 and `n`. Otherwise (for `n > 1`) it is **composite**.

These are the *only* facts you may use about evenness/oddness/primality in §4.1 proofs. You cannot yet assume "the sum of two evens is even" — that has to be proved (and is the very thing you'll prove).

In [ ]:
def is_even(n): return n % 2 == 0
def is_odd(n):  return n % 2 != 0

def is_prime(n):
    if n < 2: return False
    if n < 4: return True
    if n % 2 == 0: return False
    for d in range(3, isqrt(n) + 1, 2):
        if n % d == 0:
            return False
    return True

def witness_even_form(n):
    return n // 2 if n % 2 == 0 else None

def witness_odd_form(n):
    return (n - 1) // 2 if n % 2 != 0 else None

# Examples from Epp 4.1.1
print("Epp Example 4.1.1 checks:")
print(f"  Is 0 even?         {is_even(0)}   (since 0 = 2·{witness_even_form(0)})")
print(f"  Is -301 odd?       {is_odd(-301)}  (since -301 = 2·({witness_odd_form(-301)}) + 1 = 2·(-151) + 1)")
print(f"  Is 6·a²·b even when a=2, b=3?  {is_even(6*2*2*3)}  (= 2·{witness_even_form(6*2*2*3)})")
print(f"  Is 10·a + 8·b + 1 odd when a=2, b=5?  {is_odd(10*2 + 8*5 + 1)}  (= 2·{witness_odd_form(10*2 + 8*5 + 1)} + 1)")

## 4.1.2 — Disproof by counterexample

To **disprove** a statement of the form `∀x ∈ D, P(x) → Q(x)`, find a single `x` for which `P(x)` is true but `Q(x)` is false. One counterexample is enough.

Below: a **conjecture lab**. Pick a statement, choose a search range, and the notebook reports whether a counterexample was found.

⚠️ Remember: **"no counterexample found in [-N, N]" does NOT prove the statement is true** for all integers. It only rules out counterexamples in that range.

In [ ]:
# Each conjecture: (statement_text, P, Q, is_true_in_general, note)
# P and Q are predicates on a single integer n.
conjectures = {
    'For all integers n, if n is odd then (n-1)/2 is odd.':
        (lambda n: is_odd(n),
         lambda n: is_odd((n - 1) // 2),
         False,
         'FALSE — counterexample n=5: (5-1)/2 = 2 is even. This is Epp §4.1 exercise 12.'),
    'For all integers n, if n is odd then n² is odd.':
        (lambda n: is_odd(n),
         lambda n: is_odd(n * n),
         True,
         'TRUE — this is Epp §4.1 exercise 28. Search alone cannot establish it; a proof is needed.'),
    'For all integers m ≥ 3, m² - 1 is prime.':
        (lambda n: n >= 3,
         lambda n: is_prime(n * n - 1),
         False,
         'FALSE — counterexample m=3: 9-1=8 is composite. Epp §4.1 exercise 35.'),
    'For all integers n, 2n² - 5n + 2 is prime.':
        (lambda n: True,
         lambda n: is_prime(2*n*n - 5*n + 2),
         False,
         'FALSE — many counterexamples. Composite when 2n²-5n+2 factors (e.g., n=0 gives 2, prime; n=1 gives -1, not prime).'),
    'For all integers n, n² - n + 11 is prime.':
        (lambda n: True,
         lambda n: is_prime(n*n - n + 11),
         False,
         'FALSE in general — works for n=1..10 (Epp §4.1.18) but fails at n=11: 11²-11+11 = 121 = 11². A famous trap.'),
    'For all positive integers m and n, m·n ≥ m + n.':
        (lambda n: n >= 1,
         lambda n: n * 1 >= n + 1,  # fixing m=1
         False,
         'FALSE — set m=1. Then m·n = n but m+n = n+1. So m·n < m+n. Epp §4.1 exercise 11 (variant).'),
}

conj_dd = widgets.Dropdown(options=list(conjectures.keys()), description='Statement:', style={'description_width': '80px'}, layout=widgets.Layout(width='700px'))
range_slider = widgets.IntSlider(value=20, min=5, max=500, step=5, description='Search |n| ≤', style={'description_width': '80px'}, layout=widgets.Layout(width='400px'))
conj_out = widgets.Output()

def hunt(change=None):
    P, Q, truth, note = conjectures[conj_dd.value]
    N = range_slider.value
    cex = []
    checked = 0
    for n in range(-N, N + 1):
        if P(n):
            checked += 1
            if not Q(n):
                cex.append(n)
                if len(cex) >= 5:
                    break
    with conj_out:
        conj_out.clear_output()
        print(f"Statement: {conj_dd.value}")
        print(f"Searched integers in [{-N}, {N}]; {checked} satisfied the hypothesis.")
        print()
        if cex:
            print(f"✗ Counterexamples found: {cex}")
            n0 = cex[0]
            print(f"  e.g. n = {n0}: hypothesis P({n0}) is True, but conclusion Q({n0}) is False.")
            print(f"  → The statement is DISPROVED.")
        else:
            print(f"  No counterexample found in this range.")
            print(f"  ⚠ This does NOT prove the statement. Larger ranges may still reveal counterexamples,")
            print(f"     and even searching all integers is impossible. A proof is required.")
        print()
        print(f"Reference: {note}")

conj_dd.observe(hunt, names='value')
range_slider.observe(hunt, names='value')
display(conj_dd, range_slider, conj_out)
hunt()

## 4.1.3 — Direct proof and the generic-particular method

To prove `∀x ∈ D, P(x) → Q(x)` directly:

1. Suppose `x` is a particular but arbitrarily chosen element of `D` for which `P(x)` is true.
2. Show, using definitions and known facts, that `Q(x)` must also be true.

The variable name is for *anybody* in `D` — that's why the argument generalises.

**Worked example (Epp Theorem 4.1.1 style).** *Claim:* The sum of any two even integers is even.

**Proof.** Suppose `m` and `n` are any two even integers. By definition of even, `m = 2r` and `n = 2s` for some integers `r` and `s`. Then
$$m + n = 2r + 2s = 2(r + s).$$
Let `t = r + s`. Then `t` is an integer (sums of integers are integers). So `m + n = 2t`, which means `m + n` is even by definition.  ∎

Below: a calculator-style demo that **illustrates** the algebraic step (and lets you check sample inputs), without pretending to *prove* the claim.

In [ ]:
m_in = widgets.IntText(value=6, description='m (even):', style={'description_width': '80px'}, layout=widgets.Layout(width='200px'))
n_in = widgets.IntText(value=10, description='n (even):', style={'description_width': '80px'}, layout=widgets.Layout(width='200px'))
sum_out = widgets.Output()

def show_sum(change=None):
    m, n = m_in.value, n_in.value
    with sum_out:
        sum_out.clear_output()
        if not is_even(m) or not is_even(n):
            print("Both inputs must be even for this demonstration.")
            return
        r = m // 2
        s = n // 2
        total = m + n
        t = r + s
        print(f"m = {m} = 2·{r}      (so r = {r})")
        print(f"n = {n} = 2·{s}      (so s = {s})")
        print(f"m + n = 2·{r} + 2·{s} = 2·({r} + {s}) = 2·{t}")
        print(f"        = {total}     ✓ even, witnessed by t = r + s = {t}")
        print()
        print("This illustrates ONE case. The proof above covers ALL cases by leaving r, s symbolic.")

m_in.observe(show_sum, names='value')
n_in.observe(show_sum, names='value')
display(widgets.HBox([m_in, n_in]), sum_out)
show_sum()

### Practice exercises — Section 4.1
*(Adapted from Epp §4.1, suggested questions: 12, 28, 36.)*

**Exercise 4.1.A.** Disprove by counterexample:
$$\forall \text{ integers } n,\;\; \text{if } n \text{ is odd then } \tfrac{n-1}{2} \text{ is odd.}$$
Use the conjecture lab above to find one, then explain in writing what makes it a counterexample.

In [ ]:
sol_41a = widgets.Output()
btn_41a = widgets.Button(description='Show solution', button_style='info')

def show_41a(b):
    with sol_41a:
        sol_41a.clear_output()
        print("Counterexample: n = 5.")
        print()
        print("Hypothesis check:  is 5 odd?  Yes — 5 = 2·2 + 1.")
        print("Conclusion check:  (5-1)/2 = 4/2 = 2. Is 2 odd?  No — 2 = 2·1.")
        print()
        print("So n = 5 makes the hypothesis TRUE and the conclusion FALSE.")
        print("One such counterexample is enough to disprove a universal statement.")
        print()
        print("Other counterexamples in small range: n = 1 ((1-1)/2 = 0, even), n = 9 ((9-1)/2 = 4, even).")

btn_41a.on_click(show_41a)
display(btn_41a, sol_41a)

**Exercise 4.1.B.** Prove directly: *For all integers n, if n is odd then n² is odd.*

Write a short proof using only the definitions of odd and the fact that sums and products of integers are integers.

In [ ]:
sol_41b = widgets.Output()
btn_41b = widgets.Button(description='Show solution', button_style='info')

def show_41b(b):
    with sol_41b:
        sol_41b.clear_output()
        print("Proof (direct):")
        print()
        print("Suppose n is any [particular but arbitrarily chosen] odd integer.")
        print("[We must show that n² is odd.]")
        print()
        print("By definition of odd, n = 2k + 1 for some integer k.")
        print("By substitution and algebra,")
        print()
        print("    n² = (2k + 1)²")
        print("       = 4k² + 4k + 1")
        print("       = 2(2k² + 2k) + 1.")
        print()
        print("Let t = 2k² + 2k. Then t is an integer because products and sums of integers")
        print("are integers. Hence n² = 2t + 1, where t is an integer, and so n² is odd by")
        print("definition of odd. [This is what was to be shown.] ∎")
        print()
        print("Sample check (n = 7):")
        n = 7
        k = (n - 1) // 2
        t = 2*k*k + 2*k
        print(f"    k = {k}, t = 2k²+2k = {t}, n² = 2·{t} + 1 = {2*t + 1} = {n*n} ✓")

btn_41b.on_click(show_41b)
display(btn_41b, sol_41b)

**Exercise 4.1.C.** Disprove the existential claim by exhaustion of small cases, or prove it true with a witness:

$$\exists \text{ an integer } n \text{ such that } 6n^2 + 27 \text{ is prime.}$$


In [ ]:
sol_41c = widgets.Output()
btn_41c = widgets.Button(description='Show solution', button_style='info')

def show_41c(b):
    with sol_41c:
        sol_41c.clear_output()
        print("The statement is FALSE — but here it cannot be shown by a single counterexample,")
        print("because the claim is existential. We must show NO n works.")
        print()
        print("Proof. Suppose, for contradiction, that 6n² + 27 is prime for some integer n.")
        print("Notice that 6n² + 27 = 3(2n² + 9). Since 2n² + 9 is an integer ≥ 9 > 1, the")
        print("expression 6n² + 27 has 3 as a divisor and 6n² + 27 > 3. So it has a divisor")
        print("other than 1 and itself — it is composite, not prime. Contradiction.")
        print()
        print("Hence there is no integer n for which 6n² + 27 is prime. ∎")
        print()
        print("Sample values:")
        for n in range(0, 6):
            v = 6*n*n + 27
            print(f"    n = {n}: 6n² + 27 = {v} = 3 · {v // 3}  (composite)")

btn_41c.on_click(show_41c)
display(btn_41c, sol_41c)

---
# Section 4.2 — Direct proof and counterexample II: rational numbers

**Rational** (*نسبي*). A real number `r` is **rational** iff `r = a/b` for some integers `a, b` with `b ≠ 0`.

**Irrational** (*غير نسبي*). A real number is **irrational** iff it is not rational.

**Theorem 4.2.2** (Epp). *The sum of any two rational numbers is rational.*

**Proof.** Suppose `r` and `s` are rational. By definition, `r = a/b` and `s = c/d` for some integers `a, b, c, d` with `b ≠ 0` and `d ≠ 0`. Then
$$r + s = \frac{a}{b} + \frac{c}{d} = \frac{ad + bc}{bd}.$$
Let `p = ad + bc` and `q = bd`. Both `p` and `q` are integers (sums and products of integers are integers), and `q ≠ 0` by the zero-product property (`b, d` are both nonzero). So `r + s = p/q` with integers `p, q` and `q ≠ 0`, hence rational by definition.  ∎

In [ ]:
# Rational closure demonstrator (illustrates, does not prove).
import random

def demo_rational_sum():
    a, b = random.randint(-9, 9), random.randint(1, 9)
    c, d = random.randint(-9, 9), random.randint(1, 9)
    r = Fraction(a, b)
    s = Fraction(c, d)
    total = r + s
    p = a*d + b*c
    q = b*d
    print(f"r = {a}/{b} = {r}")
    print(f"s = {c}/{d} = {s}")
    print(f"r + s = (a·d + b·c) / (b·d) = ({a}·{d} + {b}·{c}) / ({b}·{d}) = {p}/{q}")
    print(f"     = {total}  (reduced form)")
    print(f"Result is rational: numerator {p} and denominator {q} are integers, q ≠ 0. ✓")
    print()

btn_demo = widgets.Button(description='Try another random pair', button_style='primary')
demo_out = widgets.Output()

def on_demo(b):
    with demo_out:
        demo_out.clear_output()
        demo_rational_sum()

btn_demo.on_click(on_demo)
display(btn_demo, demo_out)

with demo_out:
    demo_rational_sum()

## 4.2.1 — The sum of a rational and an irrational is irrational

**Theorem 4.6.3** (proved by contradiction — anticipating §4.6).

*Proof sketch.* Suppose `r` is rational, `s` is irrational, but `r + s` were rational. Then `r = a/b` and `r + s = c/d` for integers `a, b, c, d` with `b, d ≠ 0`. So
$$s = (r + s) - r = \frac{c}{d} - \frac{a}{b} = \frac{bc - ad}{bd},$$
which is a quotient of integers with nonzero denominator — hence `s` is rational. Contradiction. So `r + s` must be irrational.  ∎

This is a *closure* result phrased negatively. The interactive cell below illustrates one direction by sampling irrationals like `√2` (approximated as float for display only — the proof above does not depend on a numerical approximation).

In [ ]:
import math

irr_dd = widgets.Dropdown(
    options=[('√2', math.sqrt(2)), ('π', math.pi), ('e', math.e), ('√3', math.sqrt(3))],
    description='Irrational s:',
    style={'description_width': '90px'},
    layout=widgets.Layout(width='300px')
)
r_a = widgets.IntText(value=1, description='a:', layout=widgets.Layout(width='150px'))
r_b = widgets.IntText(value=2, description='b (≠0):', layout=widgets.Layout(width='150px'))
rs_out = widgets.Output()

def show_rs(change=None):
    a = r_a.value
    b = r_b.value
    s = irr_dd.value
    with rs_out:
        rs_out.clear_output()
        if b == 0:
            print("b must be nonzero.")
            return
        r = a / b
        print(f"r = {a}/{b} = {r:.6f}   (rational)")
        print(f"s = {irr_dd.label} ≈ {s:.6f}   (irrational)")
        print(f"r + s ≈ {r + s:.6f}")
        print()
        print("The theorem guarantees r + s is irrational — no integer p, q can give r + s = p/q exactly.")
        print("Numerical evidence cannot prove this; the contradiction argument above does.")

r_a.observe(show_rs, names='value')
r_b.observe(show_rs, names='value')
irr_dd.observe(show_rs, names='value')
display(widgets.HBox([r_a, r_b, irr_dd]), rs_out)
show_rs()

### Practice exercises — Section 4.2
*(Adapted from Epp §4.2, suggested questions: 16, 17, 22.)*

**Exercise 4.2.A.** True or false: *The difference of any two rational numbers is a rational number.* Prove or disprove.

In [ ]:
sol_42a = widgets.Output()
btn_42a = widgets.Button(description='Show solution', button_style='info')

def show_42a(b):
    with sol_42a:
        sol_42a.clear_output()
        print("TRUE. Proof:")
        print()
        print("Suppose r and s are rational. By definition, r = a/b and s = c/d for some")
        print("integers a, b, c, d with b ≠ 0 and d ≠ 0. Then")
        print()
        print("    r − s = a/b − c/d = (ad − bc) / (bd).")
        print()
        print("Let p = ad − bc and q = bd. Then p and q are integers because differences and")
        print("products of integers are integers, and q ≠ 0 by the zero-product property.")
        print("Hence r − s = p/q where p, q are integers and q ≠ 0, so r − s is rational.  ∎")
        print()
        print("Sample check:")
        a, b, c, d = 3, 4, 1, 6
        print(f"    r = {a}/{b}, s = {c}/{d}")
        print(f"    r − s = ({a}·{d} − {b}·{c}) / ({b}·{d}) = {a*d - b*c}/{b*d}")
        print(f"          = {Fraction(a*d - b*c, b*d)}  ✓ rational")

btn_42a.on_click(show_42a)
display(btn_42a, sol_42a)

**Exercise 4.2.B.** Prove: *The quotient of any two rational numbers, where the divisor is nonzero, is a rational number.* (Why does the nonzero condition matter?)

In [ ]:
sol_42b = widgets.Output()
btn_42b = widgets.Button(description='Show solution', button_style='info')

def show_42b(b):
    with sol_42b:
        sol_42b.clear_output()
        print("Proof:")
        print()
        print("Suppose r and s are rational numbers with s ≠ 0. By definition,")
        print("r = a/b and s = c/d for some integers a, b, c, d with b ≠ 0 and d ≠ 0.")
        print("Since s ≠ 0, also c ≠ 0 (otherwise s = 0/d = 0).")
        print()
        print("Then")
        print("    r / s = (a/b) / (c/d) = (a · d) / (b · c) = ad / bc.")
        print()
        print("Let p = ad and q = bc. Both p and q are integers (products of integers are")
        print("integers). And q = bc ≠ 0 since b ≠ 0 and c ≠ 0 (zero-product property).")
        print("Hence r/s = p/q with integers p, q and q ≠ 0, so r/s is rational. ∎")
        print()
        print("Why nonzero matters: if s = 0 we would have c = 0, making q = bc = 0, and r/s")
        print("would be undefined. The closure result only holds when division is defined.")

btn_42b.on_click(show_42b)
display(btn_42b, sol_42b)

**Exercise 4.2.C.** True or false: *If `a` is any odd integer, then `a² + a` is even.* Prove or disprove.

In [ ]:
sol_42c = widgets.Output()
btn_42c = widgets.Button(description='Show solution', button_style='info')

def show_42c(b):
    with sol_42c:
        sol_42c.clear_output()
        print("TRUE. Proof:")
        print()
        print("Suppose a is any odd integer. By definition of odd, a = 2k + 1 for some integer k.")
        print("Then")
        print("    a² + a = (2k + 1)² + (2k + 1)")
        print("           = 4k² + 4k + 1 + 2k + 1")
        print("           = 4k² + 6k + 2")
        print("           = 2(2k² + 3k + 1).")
        print()
        print("Let t = 2k² + 3k + 1. Then t is an integer (sums and products of integers are")
        print("integers). So a² + a = 2t, hence even by definition.  ∎")
        print()
        print("Insight: a² + a = a(a + 1) is the product of two consecutive integers, so one")
        print("of them is even — making the product even regardless of a's parity. The proof")
        print("above just handles the case where a is odd; the general fact is even stronger.")
        print()
        print("Sample check:")
        for a in [3, 5, 7, -1]:
            v = a*a + a
            print(f"    a = {a:3}: a² + a = {v:3} = 2·{v // 2}")

btn_42c.on_click(show_42c)
display(btn_42c, sol_42c)

---
# Section 4.3 — Direct proof and counterexample III: divisibility

**Definition.** If `n` and `d` are integers with `d ≠ 0`, then `d` **divides** `n` (written `d | n`) iff `n = d · k` for some integer `k`.

Synonyms: "`n` is divisible by `d`", "`d` is a factor of `n`", "`n` is a multiple of `d`".

In [ ]:
def divides(d, n):
    return d != 0 and n % d == 0

def witness_factor(d, n):
    return n // d if divides(d, n) else None

d_in = widgets.IntText(value=3, description='d:', layout=widgets.Layout(width='150px'))
n_in = widgets.IntText(value=24, description='n:', layout=widgets.Layout(width='150px'))
div_out = widgets.Output()

def show_div(change=None):
    d, n = d_in.value, n_in.value
    with div_out:
        div_out.clear_output()
        if d == 0:
            print("d must be nonzero.")
            return
        if divides(d, n):
            k = witness_factor(d, n)
            print(f"d = {d}, n = {n}")
            print(f"{d} | {n} is TRUE.  Witness: n = d · k = {d} · ({k}) = {d*k}.")
        else:
            q, r = n // d, n % d
            print(f"d = {d}, n = {n}")
            print(f"{d} | {n} is FALSE.  Closest: {n} = {d}·{q} + {r}, remainder {r} ≠ 0.")

d_in.observe(show_div, names='value')
n_in.observe(show_div, names='value')
display(widgets.HBox([d_in, n_in]), div_out)
show_div()

## 4.3.1 — Properties of divisibility (Epp §4.3)

**Transitivity of divisibility.** For all integers `a, b, c`: if `a | b` and `b | c`, then `a | c`.

**Proof.** Suppose `a | b` and `b | c`. By definition, `b = a · r` and `c = b · s` for some integers `r, s`. Then `c = b · s = (a · r) · s = a · (r · s)`. Let `t = r · s`. Then `t` is an integer (products of integers are integers), so `c = a · t`, meaning `a | c`. ∎

**Divisibility by a linear combination.** For all integers `a, b, c`: if `a | b` and `a | c`, then `a | (b + c)`, `a | (b − c)`, and more generally `a | (bx + cy)` for any integers `x, y`.

**Proof of `a | (b + c)`.** Suppose `a | b` and `a | c`. Then `b = a·r` and `c = a·s`. So `b + c = a·r + a·s = a·(r + s)`. Let `t = r + s` (an integer). Then `b + c = a·t`, so `a | (b + c)`.  ∎

In [ ]:
# Transitivity demo
a_in = widgets.IntText(value=3, description='a:', layout=widgets.Layout(width='130px'))
b_in = widgets.IntText(value=12, description='b:', layout=widgets.Layout(width='130px'))
c_in = widgets.IntText(value=60, description='c:', layout=widgets.Layout(width='130px'))
trans_out = widgets.Output()

def show_trans(change=None):
    a, b, c = a_in.value, b_in.value, c_in.value
    with trans_out:
        trans_out.clear_output()
        if a == 0 or b == 0:
            print("a and b must be nonzero.")
            return
        ab = divides(a, b)
        bc = divides(b, c)
        ac = divides(a, c)
        print(f"Check: a | b ?  {a} | {b}: {ab}" + (f"  ({b} = {a}·{b//a})" if ab else ""))
        print(f"Check: b | c ?  {b} | {c}: {bc}" + (f"  ({c} = {b}·{c//b})" if bc else ""))
        print(f"Check: a | c ?  {a} | {c}: {ac}" + (f"  ({c} = {a}·{c//a})" if ac else ""))
        print()
        if ab and bc:
            r = b // a
            s = c // b
            t = r * s
            print(f"Transitivity holds: r = {r}, s = {s}, t = r·s = {t},")
            print(f"so c = a·t = {a}·{t} = {a*t}.  ✓")
            print()
            print("The proof works for ANY integers a, b, c with these divisibility properties.")
        else:
            print("Both hypotheses (a|b and b|c) need to hold to apply the theorem.")

for w in (a_in, b_in, c_in):
    w.observe(show_trans, names='value')
display(widgets.HBox([a_in, b_in, c_in]), trans_out)
show_trans()

## 4.3.2 — The unique factorisation theorem

**Theorem (Unique Factorisation of Integers, Epp §4.3).** Every integer `n > 1` can be written as a product of primes, and this factorisation is unique up to the order of the factors.

This is the foundation of number theory — every positive integer has a *fingerprint* in terms of primes.

In [ ]:
def prime_factorise(n):
    if n < 2:
        return None
    factors = {}
    d = 2
    while d * d <= n:
        while n % d == 0:
            factors[d] = factors.get(d, 0) + 1
            n //= d
        d += 1
    if n > 1:
        factors[n] = factors.get(n, 0) + 1
    return factors

def factor_string(factors):
    parts = []
    for p in sorted(factors):
        e = factors[p]
        parts.append(f"{p}" if e == 1 else f"{p}^{e}")
    return " · ".join(parts)

fact_in = widgets.IntText(value=360, description='n:', layout=widgets.Layout(width='200px'))
fact_out = widgets.Output()

def show_fact(change=None):
    n = fact_in.value
    with fact_out:
        fact_out.clear_output()
        if n < 2:
            print("Need n ≥ 2 for prime factorisation.")
            return
        f = prime_factorise(n)
        print(f"n = {n}")
        print(f"Prime factorisation: {n} = {factor_string(f)}")
        print(f"Distinct primes: {sorted(f.keys())}")
        # Verify
        product = 1
        for p, e in f.items():
            product *= p ** e
        print(f"Verification: product = {product}  ({'✓' if product == n else '✗'})")

fact_in.observe(show_fact, names='value')
display(fact_in, fact_out)
show_fact()

### Practice exercises — Section 4.3
*(Adapted from Epp §4.3, suggested questions: 13, 21, 29.)*

**Exercise 4.3.A.** Prove: *For all integers `a`, `b`, and `c`, if `a | b` and `a | c`, then `a | (b − c)`.*

In [ ]:
sol_43a = widgets.Output()
btn_43a = widgets.Button(description='Show solution', button_style='info')

def show_43a(b):
    with sol_43a:
        sol_43a.clear_output()
        print("Proof:")
        print()
        print("Suppose a, b, c are integers with a | b and a | c. By definition of divisibility,")
        print("b = a·r for some integer r, and c = a·s for some integer s.")
        print()
        print("Then")
        print("    b − c = a·r − a·s = a·(r − s).")
        print()
        print("Let t = r − s. Then t is an integer because differences of integers are integers.")
        print("So b − c = a·t, hence a | (b − c) by definition. ∎")
        print()
        print("Sample check:")
        a, r, s = 5, 4, 7
        b, c = a*r, a*s
        print(f"    a = {a}, b = a·r = {a}·{r} = {b}, c = a·s = {a}·{s} = {c}")
        print(f"    b − c = {b} − {c} = {b-c} = {a}·{r-s} = a·(r−s)  ✓")

btn_43a.on_click(show_43a)
display(btn_43a, sol_43a)

**Exercise 4.3.B.** Prove or disprove: *For all integers `a, b`, if `a | b` then `a | b²`.*

In [ ]:
sol_43b = widgets.Output()
btn_43b = widgets.Button(description='Show solution', button_style='info')

def show_43b(b):
    with sol_43b:
        sol_43b.clear_output()
        print("TRUE. Proof:")
        print()
        print("Suppose a and b are integers with a | b. By definition, b = a·k for some integer k.")
        print()
        print("    b² = (a·k)² = a² · k² = a · (a · k²).")
        print()
        print("Let t = a · k². Then t is an integer (products of integers are integers).")
        print("So b² = a · t, hence a | b² by definition. ∎")
        print()
        print("Note: this is a step toward proving 'a | b ⇒ a | b^n for all positive integers n',")
        print("which is provable by induction (Chapter 5).")

btn_43b.on_click(show_43b)
display(btn_43b, sol_43b)

**Exercise 4.3.C.** Disprove by counterexample: *For all integers `a, b, c`, if `a | (b · c)` then `a | b` or `a | c`.*

(This is the *converse* of a true theorem — but the converse fails!)

In [ ]:
sol_43c = widgets.Output()
btn_43c = widgets.Button(description='Show solution', button_style='info')

def show_43c(b):
    with sol_43c:
        sol_43c.clear_output()
        print("FALSE. Counterexample: a = 6, b = 4, c = 9.")
        print()
        print(f"    b · c = 4 · 9 = 36, and 6 | 36 since 36 = 6·6.")
        print(f"    Does 6 | 4?  No, since 4 = 6·0 + 4 (remainder 4).")
        print(f"    Does 6 | 9?  No, since 9 = 6·1 + 3 (remainder 3).")
        print()
        print("So a | (b·c) holds, but neither a | b nor a | c holds. The statement is disproved.")
        print()
        print("Why does this happen? Because 6 = 2·3 splits its prime factors between b (which has")
        print("a factor of 2) and c (which has a factor of 3). The product gathers both factors,")
        print("but neither b nor c alone supplies all of them.")
        print()
        print("The statement IS true when a is prime (Euclid's lemma). For composite a, it can fail.")
        print()
        print("More counterexamples found by search:")
        cex = []
        for a in range(4, 20):
            for b in range(1, 15):
                for c in range(1, 15):
                    if (b*c) % a == 0 and b % a != 0 and c % a != 0:
                        cex.append((a, b, c))
                        if len(cex) >= 5:
                            break
                if len(cex) >= 5:
                    break
            if len(cex) >= 5:
                break
        for a, b, c in cex:
            print(f"    a = {a:2}, b = {b:2}, c = {c:2}: bc = {b*c:3}, {a} | {b*c} but {a}∤{b} and {a}∤{c}")

btn_43c.on_click(show_43c)
display(btn_43c, sol_43c)

---
# Section 4.4 — Quotient-remainder theorem, div/mod, and division into cases

## 4.4.1 — The quotient-remainder theorem (Epp Theorem 4.4.1)

For every integer `n` and positive integer `d`, there exist **unique** integers `q` and `r` such that
$$n = d \cdot q + r \quad \text{and} \quad 0 \le r < d.$$

`q` is the **quotient** (`n div d`), and `r` is the **remainder** (`n mod d`).

⚠️ **Watch out for negative `n`.** Python's `%` already returns a non-negative remainder for positive `d` — this matches Epp's `mod`. But many languages (C, C++, Java) return a remainder with the same sign as `n` for negative inputs. The cell below uses Python's convention, which matches Epp's `mod` directly.

In [ ]:
def epp_div(n, d):
    return n // d  # Python floor division — matches Epp's div when d > 0

def epp_mod(n, d):
    return n % d   # Non-negative for d > 0 in Python

n_qr = widgets.IntText(value=54, description='n:', layout=widgets.Layout(width='200px'))
d_qr = widgets.IntText(value=4, description='d (>0):', layout=widgets.Layout(width='200px'))
qr_out = widgets.Output()

def show_qr(change=None):
    n, d = n_qr.value, d_qr.value
    with qr_out:
        qr_out.clear_output()
        if d <= 0:
            print("d must be positive.")
            return
        q, r = epp_div(n, d), epp_mod(n, d)
        print(f"n = {n}, d = {d}")
        print(f"n div d = {q}")
        print(f"n mod d = {r}")
        print()
        print(f"Verification: d·q + r = {d}·{q} + {r} = {d*q + r}")
        print(f"              0 ≤ r < d: 0 ≤ {r} < {d}  ({'✓' if 0 <= r < d else '✗'})")
        print(f"              n = d·q + r:  {n} = {d*q + r}  ({'✓' if n == d*q + r else '✗'})")
        print()
        print(f"In plain words: '{n} divided by {d} gives quotient {q} and remainder {r}.'")

n_qr.observe(show_qr, names='value')
d_qr.observe(show_qr, names='value')
display(widgets.HBox([n_qr, d_qr]), qr_out)
show_qr()

# Sanity: match Epp Example 4.4.1
print("\nEpp Example 4.4.1 check:")
for n, d in [(54, 4), (-54, 4), (54, 70)]:
    q, r = epp_div(n, d), epp_mod(n, d)
    print(f"  n = {n:4}, d = {d}: q = {q:3}, r = {r:3}, n = d·q + r = {d*q + r}")

## 4.4.2 — Parity classes and division into cases

The quotient-remainder theorem with `d = 2` gives: every integer is of the form `2q` (even) or `2q + 1` (odd). With `d = 3`, every integer is `3q`, `3q+1`, or `3q+2`. With `d = 4`, four classes. And so on.

This is the basis of **proof by division into cases** (*البرهان بالحالات*): split the domain into mutually exclusive classes by remainder, prove the claim in each class, conclude it holds for all integers.

In [ ]:
mod_slider = widgets.IntSlider(value=3, min=2, max=8, step=1, description='d (modulus):', style={'description_width': '100px'}, layout=widgets.Layout(width='400px'))
range_n = widgets.IntSlider(value=10, min=5, max=30, step=1, description='Show |n| ≤', style={'description_width': '100px'}, layout=widgets.Layout(width='400px'))
classes_out = widgets.Output()

def show_classes(change=None):
    d = mod_slider.value
    N = range_n.value
    classes = {r: [] for r in range(d)}
    for n in range(-N, N + 1):
        classes[n % d].append(n)
    with classes_out:
        classes_out.clear_output()
        print(f"Partitioning integers in [{-N}, {N}] by remainder mod {d}:")
        print()
        for r in range(d):
            label = f"Class {r}  (n = {d}q + {r})" if r > 0 else f"Class 0  (n = {d}q)"
            print(f"  {label:30}: {classes[r]}")
        print()
        print(f"Every integer falls into exactly ONE class.  Total classes = {d}.")

mod_slider.observe(show_classes, names='value')
range_n.observe(show_classes, names='value')
display(mod_slider, range_n, classes_out)
show_classes()

## 4.4.3 — Worked theorem: the square of any integer has the form `4k` or `4k+1`

**(Epp §4.4 exercise 37.)** *Claim:* For every integer `n`, `n² = 4k` or `n² = 4k + 1` for some integer `k`.

**Proof.** Suppose `n` is any integer. By the quotient-remainder theorem (`d = 2`), either `n = 2q` (even) or `n = 2q + 1` (odd) for some integer `q`.

**Case 1: `n = 2q`.**
$$n^2 = (2q)^2 = 4q^2.$$
Let `k = q²`. Then `k` is an integer, and `n² = 4k`.

**Case 2: `n = 2q + 1`.**
$$n^2 = (2q + 1)^2 = 4q^2 + 4q + 1 = 4(q^2 + q) + 1.$$
Let `k = q² + q`. Then `k` is an integer, and `n² = 4k + 1`.

Both cases give the required form.  ∎

In [ ]:
sq_slider = widgets.IntSlider(value=15, min=5, max=40, step=1, description='Check |n| ≤', style={'description_width': '90px'}, layout=widgets.Layout(width='400px'))
sq_out = widgets.Output()

def show_squares(change=None):
    N = sq_slider.value
    with sq_out:
        sq_out.clear_output()
        print(f"For each n in [{-N}, {N}], compute n² mod 4 and identify the case:")
        print()
        print(f"  {'n':>4} {'n²':>6}  n² mod 4   form (k = ?)")
        print(f"  {'─'*4} {'─'*6}  {'─'*8}   {'─'*30}")
        for n in range(-N, N + 1):
            sq = n * n
            r = sq % 4
            if n % 2 == 0:
                q = n // 2
                k = q * q
                form = f"4·{k} = {4*k}    (case 1, n = 2·{q})"
            else:
                q = (n - 1) // 2
                k = q*q + q
                form = f"4·{k} + 1 = {4*k + 1}    (case 2, n = 2·{q}+1)"
            mark = '' if r in (0, 1) else '  ← unexpected!'
            print(f"  {n:>4} {sq:>6}     {r}       {form}{mark}")
        print()
        unexpected = [n for n in range(-N, N+1) if (n*n) % 4 not in (0, 1)]
        if unexpected:
            print(f"⚠ Unexpected residues for: {unexpected}  (this would break the theorem)")
        else:
            print(f"✓ Every n² in this range has remainder 0 or 1 mod 4, matching the theorem.")

sq_slider.observe(show_squares, names='value')
display(sq_slider, sq_out)
show_squares()

### Practice exercises — Section 4.4
*(Adapted from Epp §4.4, suggested questions: 4, 8, 22.)*

**Exercise 4.4.A.** Find `q` and `r` (with `0 ≤ r < d`) such that `n = d·q + r`:
1. `n = 3, d = 11`
2. `n = -45, d = 11`


In [ ]:
sol_44a = widgets.Output()
btn_44a = widgets.Button(description='Show solution', button_style='info')

def show_44a(b):
    with sol_44a:
        sol_44a.clear_output()
        for n, d in [(3, 11), (-45, 11)]:
            q, r = epp_div(n, d), epp_mod(n, d)
            print(f"n = {n}, d = {d}:")
            print(f"    q = {q}, r = {r}")
            print(f"    Check: d·q + r = {d}·({q}) + {r} = {d*q} + {r} = {d*q + r}  ✓ equals n")
            print(f"    Check: 0 ≤ r < d:  0 ≤ {r} < {d}  ✓")
            print()
        print("Note for n = −45: the quotient is −5, not −4. We need r non-negative, so we must")
        print("'overshoot' below n. (−45) = 11·(−5) + 10, since 11·(−5) = −55, and −55 + 10 = −45.")

btn_44a.on_click(show_44a)
display(btn_44a, sol_44a)

**Exercise 4.4.B.** Evaluate:
- `50 div 7` and `50 mod 7`


In [ ]:
sol_44b = widgets.Output()
btn_44b = widgets.Button(description='Show solution', button_style='info')

def show_44b(b):
    with sol_44b:
        sol_44b.clear_output()
        n, d = 50, 7
        q, r = epp_div(n, d), epp_mod(n, d)
        print(f"50 div 7 = {q}")
        print(f"50 mod 7 = {r}")
        print()
        print(f"Verification: 7·{q} + {r} = {7*q + r} = 50  ✓")
        print(f"              0 ≤ {r} < 7  ✓")

btn_44b.on_click(show_44b)
display(btn_44b, sol_44b)

**Exercise 4.4.C.** Prove: *For every integer `n`, `n² + n` is even.* (Use division into cases mod 2.)

In [ ]:
sol_44c = widgets.Output()
btn_44c = widgets.Button(description='Show solution', button_style='info')

def show_44c(b):
    with sol_44c:
        sol_44c.clear_output()
        print("Proof (by division into cases):")
        print()
        print("Suppose n is any integer. By the quotient-remainder theorem with d = 2,")
        print("either n = 2q (n even) or n = 2q + 1 (n odd) for some integer q.")
        print()
        print("Case 1: n = 2q.")
        print("    n² + n = (2q)² + 2q = 4q² + 2q = 2(2q² + q).")
        print("    Let t = 2q² + q. Then t is an integer (products and sums of integers).")
        print("    So n² + n = 2t, hence even.")
        print()
        print("Case 2: n = 2q + 1.")
        print("    n² + n = (2q + 1)² + (2q + 1)")
        print("           = 4q² + 4q + 1 + 2q + 1")
        print("           = 4q² + 6q + 2")
        print("           = 2(2q² + 3q + 1).")
        print("    Let t = 2q² + 3q + 1. Then t is an integer.")
        print("    So n² + n = 2t, hence even.")
        print()
        print("In both cases n² + n is even. ∎")
        print()
        print("Quicker insight: n² + n = n(n + 1) is the product of two consecutive integers,")
        print("one of which must be even. The cases prove this without appealing to that lemma.")
        print()
        print("Sample check:")
        for n in [-3, 0, 4, 7, 10]:
            v = n*n + n
            print(f"    n = {n:3}: n² + n = {v:3} = 2·{v // 2}")

btn_44c.on_click(show_44c)
display(btn_44c, sol_44c)

---
# Recap — Chapter 4

You can now:

- Use the definitions of **even**, **odd**, **prime**, and **composite** to justify and write short direct proofs.
- Distinguish **disproof by counterexample** (one example suffices) from **proof by example** (insufficient — except for existential claims).
- Apply the **generic-particular method**: suppose `x` is arbitrary in `D`, derive the conclusion.
- Prove closure properties of **rational numbers** (sum, difference, product, quotient).
- Use the **definition of divisibility** and prove its properties (transitivity, linear combinations).
- Use the **unique factorisation theorem** as a tool.
- Apply the **quotient-remainder theorem** to compute `div` and `mod`.
- Use **division into cases** by remainder to prove statements about all integers (e.g. *every integer's square is 4k or 4k+1*).

### What we deliberately did NOT do

We did not "prove" anything by exhaustive computer checking. The conjecture lab in §4.1 and the verification cells throughout this notebook **illustrate** claims and **find counterexamples** when claims are false, but no amount of sampling settles a universal statement over an infinite domain. The proof techniques in this chapter — direct proof, counterexample, division into cases — are what give us real certainty.

### What's next

Course continues with later chapters of Epp (sequences and induction, set theory, functions, relations, graphs, ...). Interactive notebooks for those can follow the same template established in Chapters 2, 3, and 4: build intuition with widgets, but keep the proofs primary.